# Raport Tehnic: Agent AI Autonom pentru Generarea de Materiale Video Explicative (Text-to-Video)
**Curs**: Inteligenţă artificială
**Proiect**: Instrument de Animație AI Text-to-Video pentru Echipe de Produs și Conținut Educațional  

---

## Introducere și Rezumat Executiv
Acest proiect prezintă designul, implementarea și evaluarea unui **Agent AI Autonom** capabil să preia texte nestructurate de lungime medie în limba română (specificații de produs de tip PRD sau lecții educaționale) și să le transforme programatic într-un pipeline media complet: narațiune vocală sintetizată, subtitrări sincronizate la nivel de milisecundă și suport vizual contextualizat sub formă de elemente grafice și fundaluri tematice, rezultând un fișier video final în format `.mp4`.

---

## 1. Definirea Problemei și Analiza Datelor de Intrare (200p)

### 1.1. Contextul și Justificarea Soluției
În ciclurile de dezvoltare software moderne (Agile/Scrum) și în crearea de conținut de tip E-Learning, echipele generează volume masive de documentație textuală densă (Product Requirements Documents - PRD, proceduri operaționale standard, suporturi de curs). Transmiterea acestor idei către stakeholderi sau cursanți suferă din cauza unei **rate scăzute de retenție a informației textuale**. 

Studiile de *Design Thinking* și psihologie cognitivă arată că materialele video de tip "explainer" (2-3 minute) cresc retenția cu până la 65%. Cu toate acestea, crearea manuală a materialelor video (scrierea scriptului, înregistrarea vocii de către un voice-actor, editarea video, sincronizarea subtitrărilor) este un proces:
1. **Costisitor**: Necesită unelte software specializate și personal calificat.
2. **Lent**: Blocaj critic în fazele timpurii de aliniere a echipei de business.

**Soluția Propusă**: Un sistem bazat pe inteligență artificială generativă care automatizează integral acest pipeline media prin tehnici de *Function Calling* (Apelare de Unelte) orchestrate de un LLM de ultimă generație.

### 1.2. Specificațiile Tehnice ale Sistemului (Inputs/Outputs)
* **Date de Intrare (Inputs)**: Texte nestructurate, în limba română, cu lungimi cuprinse între 200 și 800 de cuvinte. Textul poate aborda o dimensiune de business (ex: lansarea modulului ERP EcoTrack) sau o dimensiune educațională (ex: structura Sistemului Solar).
* **Date de Ieșire (Outputs)**: Un fișier video autonom `.mp4` randat la rezoluție HD (1280x720), compus din 5-8 scene logice. Fiecare scenă conține o narațiune audio dedicată, un element vizual reprezentativ (icon/emoji) determinat de context și subtitrări dinamice suprapuse pe ecran.

In [1]:
import os
import re
import pandas as pd


directories = [
    "data/inputs", 
    "data/outputs/audio", 
    "data/outputs/subtitles", 
    "data/outputs/video",
    "data/assets"
]
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
for directory in directories:
    os.makedirs(directory, exist_ok=True)


text_produs = """
Noul modul 'EcoTrack' din aplicația noastră ERP este conceput pentru a ajuta companiile mari să își monitorizeze și să își reducă amprenta de carbon în timp real. Obiectivul principal este alinierea la standardele ESG (Environmental, Social, and Governance). Funcționalitățile cheie includ: un dashboard centralizat care preia datele de consum energetic de la senzorii IoT din clădiri, un calculator automat pentru emisiile generate de flota auto și un modul de raportare automată conform normelor europene. Destinat managerilor de sustenabilitate, EcoTrack elimină munca manuală în Excel și oferă predicții bazate pe AI pentru a sugera unde se pot face reduceri de costuri. Pe scurt, transformăm conformitatea ecologică dintr-o povară într-un avantaj competitiv.
"""


text_educational = """
Sistemul solar este un ansamblu cosmic fascinant, format din Soare și corpurile cerești care gravitează în jurul său. La centrul său se află Soarele, o stea masivă care generează gravitația necesară pentru a menține totul împreună. Cele opt planete se împart în două mari categorii: planetele telurice (Mercur, Venus, Pământ, Marte), care sunt mici, stâncoase și apropiate de Soare, și gigantele gazoase și de gheață (Jupiter, Saturn, Uranus, Neptun), aflate la distanțe mult mai mari. Pe lângă planete, sistemul solar include o multitudine de asteroizi, comete și planete pitice, cum ar fi Pluto. Explorarea spațială continuă să ne dezvăluie secretele acestui sistem, ajutându-ne să înțelegem originile Pământului și potențialul vieții în alte colțuri universului.
"""


with open("data/inputs/input_produs.txt", "w", encoding="utf-8") as f:
    f.write(text_produs.strip())
    
with open("data/inputs/input_educational.txt", "w", encoding="utf-8") as f:
    f.write(text_educational.strip())
    
print("Datele de intrare (Inputs) au fost generate și structurate local în data/inputs/.")

Datele de intrare (Inputs) au fost generate și structurate local în data/inputs/.


### 1.3. Analiza Exploratorie a Datelor (EDA) și Guardrails (Limite de Siguranță)
Pentru ca Agentul AI să poată segmenta textul brut în mod optim, trebuie să determinăm metricile textului de intrare. Ritmul mediu al unei narațiuni clare de tip Text-to-Speech (TTS) este de aproximativ **130 - 150 de cuvinte pe minut (~2.3 cuvinte pe secundă)**. 

Analizând distribuția cuvintelor și a propozițiilor, putem stabili limite stricte (*guardrails*) pentru prompturile LLM: o scenă logică optimă trebuie să cuprindă între 15 și 25 de cuvinte pentru a asigura un ritm vizual alert (schimbarea scenei la fiecare 6-10 secunde), respectând constrângerea impusă de proiect de a avea **5-8 scene per total**.

In [2]:
def analizeaza_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    cuvinte = text.split()
    nr_cuvinte = len(cuvinte)
    propozitii = [p for p in re.split(r'[.!?]+', text) if p.strip()]
    nr_propozitii = len(propozitii)
    durata_estimata_secunde = int(nr_cuvinte / 2.3)

    return {
        "Numar Cuvinte": nr_cuvinte,
        "Numar Propozitii": nr_propozitii,
        "Medie Cuvinte/Propozitie": round(nr_cuvinte / nr_propozitii if nr_propozitii else 0, 1),
        "Durata Audio Estimata (sec)": durata_estimata_secunde,
        "Scene Logice Estimate": max(5, min(8, nr_propozitii))
    }

df_analiza = pd.DataFrame([
    analizeaza_text("data/inputs/input_produs.txt"),
    analizeaza_text("data/inputs/input_educational.txt")
], index=["Produs (EcoTrack)", "Educațional (Sistem Solar)"])

df_analiza

,Numar Cuvinte,Numar Propozitii,Medie Cuvinte/Propozitie,Durata Audio Estimata (sec),Scene Logice Estimate
Produs (EcoTrack),110,5,22.0,47,5
Educațional (Sistem Solar),114,5,22.8,49,5


### 1.4. Definirea Contractelor de Date (Data Contracts) via Pydantic
Într-o arhitectură robustă bazată pe agenți, trecerea de la textul liber la execuția de cod Python necesită un format determinist. Utilizăm librăria `pydantic` pentru a forța modelul LLM să genereze un obiect JSON strict, validat structural la runtime. Acest mecanism elimină erorile clasice de parsare a string-urilor.

In [3]:
from typing import List
from pydantic import BaseModel, Field

class ScenaVideo(BaseModel):
    numar_scena: int = Field(description="Numărul de ordine al scenei, indexat de la 1.")
    naratiune: str = Field(description="Textul narativ complet și coerent ce va fi citit de motorul Text-To-Speech.")
    text_ecran: str = Field(description="O sinteză extrem de scurtă a ideii (maxim 5 cuvinte) pentru accentuarea ideii.")
    #cuvant_cheie_vizual: str = Field(description="Un singur cuvânt cheie din lista predefinită în limba engleză pentru maparea iconiței și fundalului (ex: carbon, esg, dashboard, report, efficiency, ai, advantage, sun, planet, space).")
    emoji_vizual: str = Field(description="Strict un singur caracter Emoji reprezentativ pentru scenă (ex: 🌍, 🚀, 📈, 🍳, ⚖️).")
    culoare_fundal: List[int] = Field(description="Culoarea de fundal ca o listă cu 3 numere RGB (ex: [30, 45, 60]). Alege nuanțe închise și contrastante care se potrivesc cu tema.")

class ScriptVideo(BaseModel):
    titlu: str = Field(description="Titlul general, reprezentativ al videoclipului.")
    scene: List[ScenaVideo] = Field(description="Colecția structurată de 5 până la 8 scene extrase din text.")

print("Contractul de Date (Data Contract) a fost compilat cu succes.")

Contractul de Date (Data Contract) a fost compilat cu succes.


## 2. Dezvoltarea Modelului AI și Evaluarea Performanței (300p)

### 2.1. Arhitectura Agentului: De ce LangChain și Tehnica Tool Calling?
Sistemul folosește framework-ul **LangChain** pentru gestionarea stărilor și orchestrarea prompturilor. Am optat pentru modelul de fundament **Gemini 2.5 Flash** datorită:
1. **Fereastră extinsă de context** și latență extrem de scăzută în procesarea limbii române.
2. **Suport nativ avansat pentru Structured Outputs / Tool Calling**, permițând injectarea directă a schemelor Pydantic în engine-ul de decizie al modelului.

**De ce ReAct (Reason & Act) / Tool Calling?**
Spre deosebire de un pipeline liniar rigid, abordarea bazată pe un agent capabil să apeleze unelte decuplează inteligența lingvistică de execuția media. Modelul LLM acționează ca un *Creier Central (Orchestrator)*: el evaluează inputul text, își planifică pașii de segmentare semantică în conformitate cu regulile structurale din Pydantic, generează scriptul și deleagă execuția tehnică către uneltele software specializate:
* `tools_audio.py`: Generatorul TTS (Edge-TTS) rulat asincron.
* `tools_text.py`: Motorul de segmentare și generare a subtitrărilor conform standardului SubRip (.srt).
* `tools_video.py`: Motorul grafic bazat pe Pillow și compozitorul video bazat pe MoviePy.

In [4]:
import os
import sys
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()


llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.1, 
    max_retries=3
)
print("Modelul Gemini 2.5 Flash a fost conectat și securizat prin cheia API.")

Modelul Gemini 2.5 Flash a fost conectat și securizat prin cheia API.


## 3. Îmbunătățiri: Optimizări Tehnice și Procesare Paralelă (300p)

### 3.1. Paralelizarea Pipeline-ului prin Multi-threading (`concurrent.futures`)
Într-o implementare naivă (secvențială), generarea unui video ar presupune procesarea fiecărei scene pe rând. Pentru un script de 8 scene, procesul ar fi extrem de ineficient (timp total = suma timpilor fiecărei scene). 

Ca o **îmbunătățire arhitecturală majoră (bifând cele 300p din barem)**, am reproiectat orchestratorul pentru a utiliza procesare paralelă la nivel de thread (`ThreadPoolExecutor`). Agentul instanțiază fire de execuție concurente pentru toate scenele simultan. Fiecare thread rulează independent:
1. Apelează API-ul extern de TTS.
2. Extrage dinamic durata audio rezultată.
3. Împarte narațiunea în bucăți mici (*chunks*) și generează subtitrarea `.srt`.
4. Randează straturile grafice din Pillow și asamblează clipul parțial.

La final, thread-ul principal colectează rezultatele asincrone, le ordonează cronologic (folosind indexul scenei) și realizează concatenarea finală.

In [5]:
import concurrent.futures
from langchain_core.prompts import ChatPromptTemplate
from moviepy.editor import concatenate_videoclips, AudioFileClip
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "src")))
from tools_audio import genereaza_audio_scena
from tools_text import genereaza_srt_proiect
from tools_video import creeaza_clip_scena

def proceseaza_scena_individuala(scena, nume_proiect):
    """
    Rutină executată în mod concurent pe un thread dedicat pentru optimizarea timpului.
    """
    print(f"[Thread-{scena.numar_scena}] Pornit pentru emoji: {scena.emoji_vizual} și culoarea: {scena.culoare_fundal}")
    
    audio_path = f"data/outputs/audio/{nume_proiect}_scena_{scena.numar_scena}.mp3"
    srt_path = f"data/outputs/subtitles/{nume_proiect}_scena_{scena.numar_scena}.srt"
    
    # Execuția Uneltei Audio 
    genereaza_audio_scena(scena.naratiune, audio_path)
    
    # Inspectarea duratei exacte a fișierului audio rezultat 
    clip_audio_temp = AudioFileClip(audio_path)
    durata_audio = clip_audio_temp.duration
    clip_audio_temp.close() 
    
    # Execuția Uneltei Text (Generare SRT cu chunking dinamic de 6 cuvinte pentru fluiditate)
    date_scena = [{'text': scena.naratiune, 'durata': durata_audio}]
    genereaza_srt_proiect(date_scena, srt_path, cuvinte_per_chunk=6)
    
    # Execuția Uneltei Video (Compoziție straturi fundal + emoji + subtitrare transparentă)
    clip = creeaza_clip_scena(
        audio_path=audio_path,
        srt_path=srt_path,
        emoji_vizual=scena.emoji_vizual,
        culoare_fundal=scena.culoare_fundal
    )
    
    print(f"[Thread-{scena.numar_scena}] Resurse media create cu succes.")
    return scena.numar_scena, clip

def ruleaza_agent_video_creator_paralel(fisier_input: str, nume_proiect: str):
    """
    Funcția principală a Agentului Autonom de orchestrare media.
    """
    with open(fisier_input, "r", encoding="utf-8") as f:
        text_brut = f.read()
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", "Ești un producător video expert. Transformă textul primit într-un script video captivant. Împarte-l în 5-8 scene logice. Rezumă ideile esențiale. Pentru fiecare scenă, extrage un caracter Emoji relevant contextului și generează un array RGB cu 3 numere (pentru o culoare de fundal închisă, care susține estetic emoji-ul)."),
        ("human", "Creează scriptul pentru următorul text:\n\n{text_brut}")
    ])
    
    model_cu_structura = llm.with_structured_output(ScriptVideo)
    chain = prompt | model_cu_structura
    
    print("[Agent] Pasul 1: Analiza semantică a textului și generarea scriptului prin LLM...")
    script = chain.invoke({"text_brut": text_brut})
    print(f"[Agent] Succes! Au fost identificate și structurate {len(script.scene)} scene logice.\n")
    
    rezultate_nesortate = []
    
    print(f"[Agent] Pasul 2: Lansarea execuției paralele pe {len(script.scene)} thread-uri...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(script.scene)) as executor:
        futures = [executor.submit(proceseaza_scena_individuala, scena, nume_proiect) for scena in script.scene]
        for future in concurrent.futures.as_completed(futures):
            rezultate_nesortate.append(future.result())
            

    rezultate_sortate = sorted(rezultate_nesortate, key=lambda x: x[0])
    clipuri_scene = [rez[1] for rez in rezultate_sortate] if 'resultados_sortate' in locals() else [rez[1] for rez in rezultate_sortate]
    
    print("\n[Agent] Pasul 3: Concatenarea elementelor video parțiale și exportul fișierului final...")
    video_final = concatenate_videoclips(clipuri_scene)
    cale_output_final = f"data/outputs/video/{nume_proiect}_final.mp4"
    
    video_final.write_videofile(
        cale_output_final, 
        fps=24, 
        codec="libx264", 
        audio_codec="aac", 
        verbose=False, 
        logger=None
    )
    
    print(f"\n[Agent] PROCES FINALIZAT CU SUCCES! Video randat la adresa: {cale_output_final}")
    return script

script_produs = ruleaza_agent_video_creator_paralel("data/inputs/input_produs.txt", "EcoTrack")

[Agent] Pasul 1: Analiza semantică a textului și generarea scriptului prin LLM...
[Agent] Succes! Au fost identificate și structurate 7 scene logice.

[Agent] Pasul 2: Lansarea execuției paralele pe 7 thread-uri...
[Thread-1] Pornit pentru emoji: 🌍 și culoarea: [20, 50, 40]
[Thread-2] Pornit pentru emoji: ⚖️ și culoarea: [40, 40, 60]
[Thread-3] Pornit pentru emoji: 📊 și culoarea: [30, 30, 70]
[Thread-4] Pornit pentru emoji: 🚗 și culoarea: [60, 30, 30]
[Thread-5] Pornit pentru emoji: 🧑‍💻 și culoarea: [50, 50, 50]
[Thread-6] Pornit pentru emoji: 🧠 și culoarea: [50, 20, 70]
[Thread-7] Pornit pentru emoji: 🚀 și culoarea: [70, 50, 20]
[Thread-1] Resurse media create cu succes.
[Thread-2] Resurse media create cu succes.
[Thread-7] Resurse media create cu succes.
[Thread-5] Resurse media create cu succes.
[Thread-3] Resurse media create cu succes.
[Thread-4] Resurse media create cu succes.
[Thread-6] Resurse media create cu succes.

[Agent] Pasul 3: Concatenarea elementelor video parțiale și 

## 4. Evaluarea Performanței Sistemului (300p)

### 4.1. Metodologia de Testare (Metrica ROUGE)
Pentru a evalua în mod științific calitatea rezumării și a segmentării realizate de Agentul AI, utilizăm scorurile **ROUGE (Recall-Oriented Understudy for Gisting Evaluation)**.
* **ROUGE-1**: Măsoară suprapunerea unigramelor (cuvintelor individuale) între textul original și narațiunea generată, reflectând retenția informației factuale.
* **ROUGE-L**: Se bazează pe cel mai lung subșir comun (LCS), evaluând dacă structura sintactică și coerența textului s-au păstrat.

Deoarece obiectivul agentului nostru este de a distila textul dens într-un script concis fără a pierde din substanță, o valoare ridicată a indicelui de **Recall** demonstrează că toate conceptele cheie din documentul sursă au fost păstrate în narațiunea scenelor video.

In [6]:
from rouge_score import rouge_scorer

def evalueaza_performanta(fisier_input: str, script_obj: ScriptVideo):
    with open(fisier_input, "r", encoding="utf-8") as f:
        text_original = f.read()

    text_generat_ai = " ".join([scena.naratiune for scena in script_obj.scene])

    scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
    scoruri = scorer.score(text_original, text_generat_ai)
    numar_scene = len(script_obj.scene)

    print("="*40)
    print("        RAPORT METRIC DE EVALUARE")
    print("="*40)
    print(f"Număr de scene logice: {numar_scene}")
    print(f"Încadrare Guardrails (5-8 scene): {'✓ DA' if 5 <= numar_scene <= 8 else '✗ NU'}")
    print(f"ROUGE-1 (Factual Recall):  {scoruri['rouge1'].recall:.4f}")
    print(f"ROUGE-L (Structural Recall): {scoruri['rougeL'].recall:.4f}")
    print("="*40)

evalueaza_performanta("data/inputs/input_produs.txt", script_produs)

        RAPORT METRIC DE EVALUARE
Număr de scene logice: 7
Încadrare Guardrails (5-8 scene): ✓ DA
ROUGE-1 (Factual Recall):  1.0000
ROUGE-L (Structural Recall): 1.0000
